In [47]:
%load_ext autoreload
%autoreload 2
from sindex.sources.openalex.snapshot import (
    run_openalex_sweep, 
    process_openalex_topics_for_dois, 
    process_openalex_citations_for_dois,
    create_citation_weights_table,
)
import duckdb
import os

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Test with some files from the snapshot

In [16]:
# Path configuration
config = {
    "source_dir": r"E:\OpenAlex\snapshot-light\data\works",
    "meta_out": r"C:\Users\Admin\Documents\OpenAlex\test\metadata",
    "cite_out": r"C:\Users\Admin\Documents\OpenAlex\test\citations",
     "db_path": r"C:\Users\Admin\Documents\OpenAlex\test\openalex_test.db",
    "temp": r"C:\Users\Admin\Documents\OpenAlex\test\duckdb_temp",
}

In [4]:
# Create parquet filesfrom oa works gz files
results = run_openalex_sweep(**config, max_workers=16)

if results['errors']:
    print(f"\nCompleted with errors: {len(results['errors'])}")

Syncing 13 files...
Progress: 13/13 (New: 0, Skipped: 13)

In [5]:
# Load in DuckDB
db_path = r"C:\Users\Admin\Documents\OpenAlex\test\openalex_test.db"
con = duckdb.connect(db_path)

meta_path = f"{config['meta_out']}/*.parquet"
cite_path = f"{config['cite_out']}/*.parquet"

con.execute(f"CREATE OR REPLACE TABLE works AS SELECT * FROM read_parquet('{meta_path}')")
con.execute(f"CREATE OR REPLACE TABLE citations AS SELECT * FROM read_parquet('{cite_path}')")

meta_count = con.execute("SELECT count(*) FROM works").fetchone()[0]
cite_count = con.execute("SELECT count(*) FROM citations").fetchone()[0]

print(f"Success! Loaded {meta_count:,} works and {cite_count:,} citation links.")

Loading metadata from: C:\Users\Admin\Documents\OpenAlex\test\metadata/*.parquet
Loading citations from: C:\Users\Admin\Documents\OpenAlex\test\citations/*.parquet
Success! Loaded 408 works and 689 citation links.


In [18]:
# Preview
df_sample_work = con.execute("SELECT * FROM works ORDER BY topic_id DESC LIMIT 10").df()
display(df_sample_work)
df_sample_citations = con.execute("SELECT * FROM citations ORDER BY citing_oa_id LIMIT 20").df()
display(df_sample_citations)

,oa_id,doi,pub_date,topic_id,topic_name,topic_score
0,W1547878197,10.18372/2411-264x.3.2141,2012-09-10,T13497,Hermeneutics and Narrative Identity,0.9879
1,W2288414950,10.36418/syntax-literate.v3i3.350,2018-03-27,T13497,Hermeneutics and Narrative Identity,0.9879
2,W1843317143,10.31941/delta.v1i2.482,2017-08-30,T13497,Hermeneutics and Narrative Identity,0.9879
3,W2338400371,10.11903/1002.6495.2014.087,2015-02-14,T13497,Hermeneutics and Narrative Identity,0.9879
4,W2281025132,10.18372/2412-2157.12.8295,2015-05-23,T13497,Hermeneutics and Narrative Identity,0.9879
5,W2283374966,10.2495/str030361,2003-04-10,T13497,Hermeneutics and Narrative Identity,0.9879
6,W2328865013,,1977-01-01,T13497,Hermeneutics and Narrative Identity,0.9879
7,W2303842597,,1986-02-01,T13497,Hermeneutics and Narrative Identity,0.9879
8,W2395157951,10.33772/medula.v1i1.188,2013-01-01,T13497,Hermeneutics and Narrative Identity,0.9879
9,W2190023776,10.18372/2412-2157.16.9474,2015-11-16,T13497,Hermeneutics and Narrative Identity,0.9879


,citing_oa_id,cited_oa_id
0,W139517439,W2114449116
1,W144694350,W40440957
2,W1480302865,W1966463595
3,W1480302865,W2271065608
4,W1480302865,W2236238848
5,W1480302865,W2073521201
6,W1480302865,W2338124369
7,W1480302865,W2292183954
8,W1480302865,W2044284902
9,W1480302865,W2005121720


In [19]:
target_doi = "10.11903/1002.6495.2014.087"

query = f"""
SELECT 
    -- 1. Info about the Target Paper
    w.doi AS target_doi,
    w.topic_name AS target_topic,
    w.pub_date AS target_date,
    
    -- 2. Info about the Citing Papers
    c.citing_oa_id,
    citing_meta.doi AS citing_doi,
    citing_meta.pub_date AS citing_date,
    citing_meta.topic_name AS citing_topic
FROM works w
LEFT JOIN citations c ON w.oa_id = c.cited_oa_id
LEFT JOIN works citing_meta ON c.citing_oa_id = citing_meta.oa_id
WHERE w.doi = '{target_doi}';
"""

df_results = con.execute(query).df()
display(df_results)

,target_doi,target_topic,target_date,citing_oa_id,citing_doi,citing_date,citing_topic
0,10.11903/1002.6495.2014.087,Hermeneutics and Narrative Identity,2015-02-14,None,None,None,None


In [18]:
doi_list = ['10.18372/2411-264x.3.2141', '10.36418/syntax-literate.v3i3.350']

# Connect to DuckDB (creates a file named my_data.db)
con = duckdb.connect(config["db_path"])

# Create the table using the Python list
con.execute("CREATE TABLE target_dois AS SELECT * FROM (SELECT unnest(?) AS doi)", [doi_list])

# Verify
display(con.execute("SELECT * FROM target_dois").df())

,doi
0,10.18372/2411-264x.3.2141
1,10.36418/syntax-literate.v3i3.350


In [19]:
process_openalex_topics_for_dois(
    db_path=config["db_path"], 
    meta_folder=config["meta_out"], 
    mem_limit="32GB", 
    temp_dir=config["temp"]
)

Starting scan of 13 metadata files...
Scanned: 13/13 | Elapsed: 00:00
Finalizing: Deduplicating DOIs (keeping highest topic scores)...
Done! Unique datasets found: 2 | Total Time: 0.01 min


In [20]:
display(con.execute("SELECT * FROM my_datasets_topics").df())

,oa_id,doi,pub_date,topic_id,topic_name,topic_score
0,W1547878197,10.18372/2411-264x.3.2141,2012-09-10,T13497,Hermeneutics and Narrative Identity,0.9879
1,W2288414950,10.36418/syntax-literate.v3i3.350,2018-03-27,T13497,Hermeneutics and Narrative Identity,0.9879


In [21]:
con.close()

## Actual run full OpenAlex snapshot

### Extract topics nd citations in parquet files

In [39]:
# Path configuration
config = {
    "source_dir": r"E:\OpenAlex\openalex-snapshot\data\works",
    "meta_out": r"C:\Users\Admin\Documents\OpenAlex\full-snapshot\metadata",
    "cite_out": r"C:\Users\Admin\Documents\OpenAlex\full-snapshot\citations",
    "db_path": r"C:\Users\Admin\Documents\OpenAlex\openalex-snapshot\openalex.db",
    "temp": r"C:/Users/Admin/Documents/OpenAlex/openalex-snapshot/duckdb_temp",
}

In [21]:
# Create parquet files from oa works gz files
results = run_openalex_sweep(**config, max_workers=16)

if results['errors']:
    print(f"\nCompleted with errors: {len(results['errors'])}")

Syncing 1716 files...
Progress: 1716/1716 (New: 1716, Skipped: 0)

### Get topics

In [24]:
process_openalex_topics_for_dois(
    db_path=config["db_path"], 
    meta_folder=config["meta_out"], 
    mem_limit="32GB", 
    temp_dir=config["temp"]
)

Starting scan of 1716 metadata files...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Scanned: 757/1716 | Elapsed: 12:33

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Scanned: 767/1716 | Elapsed: 12:54

...

Scanned: 1715/1716 | Elapsed: 01:20:04

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Scanned: 1716/1716 | Elapsed: 01:20:10
Finalizing: Deduplicating DOIs (keeping highest topic scores)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Done! Unique datasets found: 47,476,822 | Total Time: 82.02 min


In [55]:
con =  duckdb.connect(config["db_path"])
print("\n Dataset Topics")
row_count = con.execute("SELECT count() FROM my_datasets_topics").fetchone()[0]
print(f"Total DOIs in table: {row_count}")
display(con.execute("SELECT * FROM my_datasets_topics ORDER BY pub_date DESC LIMIT 5").df())
con.close()


 Dataset Topics
Total DOIs in table: 47476822


,oa_id,doi,pub_date,topic_id,topic_name,topic_score
0,W4393646606,10.5281/zenodo.8263053,2039-04-01,T11750,Phytoplasmas and Hemiptera pathogens,0.5456
1,W4393614663,10.5281/zenodo.6327222,2036-02-29,T13194,ICT in Developing Communities,0.3324
2,W6949782326,10.5281/zenodo.15473638,2035-05-20,None,None,NaN
3,W6893253683,10.5281/zenodo.15473322,2035-05-20,None,None,NaN
4,W6949459333,10.5281/zenodo.15473196,2035-05-20,None,None,NaN


### Get citations

In [40]:
process_openalex_citations_for_dois(
    db_path=config["db_path"], 
    cite_folder=config["cite_out"],
    meta_folder=config["meta_out"], 
    mem_limit="32GB", 
    temp_dir=config["temp"]
)

Step 1/2: Finding citations in 1716 files...
 > Progress: 1716/1716 | Elapsed: 04:58

Step 2/2: Mapping citing IDs to DOIs and Dates...
Done! Final citation count: 2,754,955 | Total Time: 72.56 min


In [44]:
con =  duckdb.connect(config["db_path"])
print("\n Dataset Citations")
row_count = con.execute("SELECT count() FROM my_datasets_citations").fetchone()[0]
print(f"Total citations in table: {row_count}")
display(con.execute("SELECT * FROM my_datasets_citations LIMIT 5").df())
con.close()


 Dataset Citations
Total citations in table: 2754955


,cited_doi,cited_oa_id,citing_oa_id,citing_doi,citation_date
0,10.4231/qf39-q924,W3215684178,W4399988887,10.1016/j.engfracmech.2024.110257,2024-06-24
1,10.5061/dryad.nvx0k6f46,W6892201352,W4412063596,10.1002/wlb3.01456,2025-07-06
2,10.57702/tuq85ufq,W2953139137,W3011708665,10.1109/iccv48922.2021.00739,2021-10-01
3,10.34894/klajfm,W6909080023,W4393070462,10.1002/wlb3.01237,2024-03-22
4,10.6092/ingv.it-dbmi15,W2773726277,W4293117205,10.5194/nhess-22-2807-2022,2022-08-26


### Add citations weight

In [50]:
create_citation_weights_table(config["db_path"])

Step 3/3: Creating weighted citation table (Defaulting NULL dates to 1.0)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Success! Table created in 12.38s
Total Citations: 2,754,955
Standard Citations (Weight 1.0): 264,605
Average Impact Score: 1.41


In [54]:
con =  duckdb.connect(config["db_path"])
print("\n Dataset Citations")
row_count = con.execute("SELECT count() FROM my_datasets_citations_weights").fetchone()[0]
print(f"Total citations in table: {row_count}")
display(con.execute("SELECT * FROM my_datasets_citations_weights ORDER BY weight ASC LIMIT 5").df())
con.close()


 Dataset Citations
Total citations in table: 2754955


,cited_doi,cited_oa_id,citing_oa_id,citing_doi,citation_date,pub_date,weight
0,10.14457/cu.the.2006.1939,W2741785518,W2461492322,,2010-01-01,2020-08-03,1.0
1,10.14457/cu.the.2006.1939,W2741785518,W2069232411,10.5539/jmr.v2n2p104,2010-04-19,2020-08-03,1.0
2,10.14457/cu.the.2006.1939,W2741785518,W4247184689,10.5539/jmr.v1n1p0,2009-02-18,2020-08-03,1.0
3,10.14457/cu.the.2006.1939,W2741785518,W1989763262,10.5539/jmr.v1n1p25,2009-02-18,2020-08-03,1.0
4,10.14291/tccon.ggg2014,W6942597812,W4213089580,10.5194/amt-2016-227,2016-08-29,2017-09-13,1.0
